In [4]:
cd projs/morflow2.0/

/ictstr01/home/icb/ghaith.mqawass/projs/morflow2.0


In [6]:
import os
import json
import torch
import pandas as pd
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader, random_split
from modules.lit_model import FlowMolBERTLitModule
from configs.data import TOK2ID,ID2TOK
import torch


local_rank = int(os.environ.get("LOCAL_RANK", 0))
print(f"Process {local_rank} using device: cuda:{local_rank}")


checkpoint_path = '/lustre/groups/ml01/workspace/ghaith.mqawass/2025_ghaith_de_novo_design/checkpoints/CrossEntropyLoss()_L=72_uniform_layers=12_dim=768_best-validity-epoch=121-validity=0.9350.ckpt'
torch.set_float32_matmul_precision('medium')

checkpoint = FlowMolBERTLitModule.load_from_checkpoint(checkpoint_path)
uncond_model = checkpoint.model

Process 0 using device: cuda:0


AssertionError: embed_dim must be divisible by num_heads

In [19]:
uncond_model

FlowMolBERT(
  (tok_emb): Embedding(173, 767)
  (pos_emb): Embedding(72, 767)
  (time_emb): Linear(in_features=1, out_features=1, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (lm_head): Linear(in_features=768, out_features=173, bias=False)
)

In [68]:
# import torch
from models.adaptive import *
from modules.cond_lit_model import CondFlowMolBERTLitModule
from configs import *
cond_module = CondFlowMolBERTLitModule(cond_dim=1024, n_layers=12,n_heads=12, time_dim=1, hidden_dim=767)

/lustre/groups/ml01/workspace/ghaith.mqawass/flow/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'loss_fn' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss_fn'])`.


In [69]:
import torch

def transfer_weights(pretrained_model, cond_model, freeze_pretrained=False):
    """
    Transfers weights from a pretrained FlowMolBERT to a conditional CondFlowMolBERT.
    
    Args:
        pretrained_model: FlowMolBERT instance (pretrained)
        cond_model: CondFlowMolBERT instance (initialized)
        freeze_pretrained (bool): whether to freeze pretrained layers after copying
    """
    # 1️⃣ Embeddings
    cond_model.tok_emb.weight.data.copy_(pretrained_model.tok_emb.weight)
    cond_model.pos_emb.weight.data.copy_(pretrained_model.pos_emb.weight)

    # Time embedding if shapes match
    if cond_model.time_emb.weight.shape == pretrained_model.time_emb.weight.shape:
        cond_model.time_emb.weight.data.copy_(pretrained_model.time_emb.weight)
        if hasattr(cond_model.time_emb, 'bias') and hasattr(pretrained_model.time_emb, 'bias'):
            cond_model.time_emb.bias.data.copy_(pretrained_model.time_emb.bias)
    
    # 2️⃣ Encoder layers
    for l_pre, l_cond in zip(pretrained_model.encoder.layers, cond_model.encoder.layers):
        # Multihead attention
        try:
            l_cond.self_attn.in_proj_weight.data.copy_(l_pre.self_attn.in_proj_weight)
            l_cond.self_attn.in_proj_bias.data.copy_(l_pre.self_attn.in_proj_bias)
            l_cond.self_attn.out_proj.weight.data.copy_(l_pre.self_attn.out_proj.weight)
            l_cond.self_attn.out_proj.bias.data.copy_(l_pre.self_attn.out_proj.bias)
        except Exception as e:
            print("⚠️ Skipping attention weights for a layer:", e)
        
        # Feedforward layers have different dimensions → cannot copy
        # LayerNorm → cannot copy directly if using AdaptiveLayerNorm

    # 3️⃣ LM head
    try:
        cond_model.lm_head.weight.data.copy_(pretrained_model.lm_head.weight)
    except Exception as e:
        print("⚠️ Skipping LM head copy:", e)

    # 4️⃣ Freeze pretrained layers if requested
    if freeze_pretrained:
        for name, param in cond_model.named_parameters():
            if 'cond_proj' not in name:  # only train new conditional layers
                param.requires_grad = False

    print("✅ Transfer completed. Conditional layers need training from scratch.")



In [70]:
transfer_weights(uncond_model, cond_module.model, freeze_pretrained=False)

✅ Transfer completed. Conditional layers need training from scratch.


In [60]:
import torch

def transfer_weights_with_adaptive_ln(pretrained_model, cond_model, freeze_pretrained=False):
    """
    Transfer pretrained weights and initialize AdaptiveLayerNorm from pretrained LayerNorm.
    """
    # 1️⃣ Embeddings
    cond_model.tok_emb.weight.data.copy_(pretrained_model.tok_emb.weight)
    cond_model.pos_emb.weight.data.copy_(pretrained_model.pos_emb.weight)

    # Time embedding if shapes match
    if cond_model.time_emb.weight.shape == pretrained_model.time_emb.weight.shape:
        cond_model.time_emb.weight.data.copy_(pretrained_model.time_emb.weight)
        if hasattr(cond_model.time_emb, 'bias') and hasattr(pretrained_model.time_emb, 'bias'):
            cond_model.time_emb.bias.data.copy_(pretrained_model.time_emb.bias)

    # 2️⃣ Encoder layers
    for l_pre, l_cond in zip(pretrained_model.encoder.layers, cond_model.encoder.layers):
        # Multihead attention
        try:
            l_cond.self_attn.in_proj_weight.data.copy_(l_pre.self_attn.in_proj_weight)
            l_cond.self_attn.in_proj_bias.data.copy_(l_pre.self_attn.in_proj_bias)
            l_cond.self_attn.out_proj.weight.data.copy_(l_pre.self_attn.out_proj.weight)
            l_cond.self_attn.out_proj.bias.data.copy_(l_pre.self_attn.out_proj.bias)
        except Exception as e:
            print("⚠️ Skipping attention weights for a layer:", e)

        # Feedforward layers: can only copy if dimensions match
        if l_cond.linear1.weight.shape == l_pre.linear1.weight.shape:
            l_cond.linear1.weight.data.copy_(l_pre.linear1.weight)
            l_cond.linear1.bias.data.copy_(l_pre.linear1.bias)
        if l_cond.linear2.weight.shape == l_pre.linear2.weight.shape:
            l_cond.linear2.weight.data.copy_(l_pre.linear2.weight)
            l_cond.linear2.bias.data.copy_(l_pre.linear2.bias)

        # Initialize AdaptiveLayerNorm from pretrained LayerNorm
        if hasattr(l_pre, 'norm1') and hasattr(l_cond, 'norm1'):
            l_cond.norm1.weight.data.copy_(l_pre.norm1.weight)
            l_cond.norm1.bias.data.copy_(l_pre.norm1.bias)
        if hasattr(l_pre, 'norm2') and hasattr(l_cond, 'norm2'):
            l_cond.norm2.weight.data.copy_(l_pre.norm2.weight)
            l_cond.norm2.bias.data.copy_(l_pre.norm2.bias)

        # Optional: zero the adaptive MLP to start with identity
        if hasattr(l_cond.norm1, 'mlp') and l_cond.norm1.mlp is not None:
            for m in l_cond.norm1.mlp:
                if isinstance(m, torch.nn.Linear):
                    torch.nn.init.zeros_(m.weight)
                    torch.nn.init.zeros_(m.bias)
        if hasattr(l_cond.norm2, 'mlp') and l_cond.norm2.mlp is not None:
            for m in l_cond.norm2.mlp:
                if isinstance(m, torch.nn.Linear):
                    torch.nn.init.zeros_(m.weight)
                    torch.nn.init.zeros_(m.bias)

    # 3️⃣ LM head
    try:
        cond_model.lm_head.weight.data.copy_(pretrained_model.lm_head.weight)
    except Exception as e:
        print("⚠️ Skipping LM head copy:", e)

    # 4️⃣ Freeze pretrained layers if requested
    if freeze_pretrained:
        for name, param in cond_model.named_parameters():
            if 'cond_proj' not in name:  # only train new conditional layers
                param.requires_grad = False

    print("✅ Transfer completed. AdaptiveLayerNorm initialized from pretrained LayerNorm.")


In [62]:
transfer_weights_with_adaptive_ln(uncond_model, cond_module.model, freeze_pretrained=False)

✅ Transfer completed. AdaptiveLayerNorm initialized from pretrained LayerNorm.


In [63]:
uncond_model

FlowMolBERT(
  (tok_emb): Embedding(173, 767)
  (pos_emb): Embedding(72, 767)
  (time_emb): Linear(in_features=1, out_features=1, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (lm_head): Linear(in_features=768, out_features=173, bias=False)
)

In [72]:
cond_module.model.encoder.layers[0].norm1.weight

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.

In [74]:
cond_module.model.encoder.layers[0].norm1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1.

In [55]:
uncond_model.tok_emb.weight

Parameter containing:
tensor([[ 0.1374, -0.2182, -0.2091,  ...,  0.1748, -1.7832,  0.2171],
        [ 0.2437,  0.7016, -0.0551,  ...,  0.1793,  1.0321, -0.5748],
        [ 0.2536,  0.9655, -0.8324,  ...,  1.6797,  0.0930, -0.7656],
        ...,
        [ 0.7335,  1.2070, -0.4143,  ..., -0.9206,  0.0563, -0.1792],
        [-0.4996,  0.0519, -0.7166,  ...,  0.0823,  0.1215,  1.0544],
        [-0.8184, -1.0275,  0.2857,  ...,  0.6500, -0.4230, -0.7864]],
       device='cuda:0', requires_grad=True)